# A9. GEO Audit Notebook — AI Search Visibility Testing

> **Related Module**: [A9 SEO/GEO](../paths/a-operators/a9-seo-geo.md)
>
> **Features**: Automatically detect your product's visibility in AI search engines
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a9-geo-audit.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q openai anthropic requests pandas

## 2. Configuration

Set up your brand information and API Key.

In [ ]:
import os
import json
from datetime import datetime

# === Your brand information ===
BRAND_NAME = 'YourBrand'  # Replace with your brand name
PRODUCT_CATEGORY = 'wireless earbuds'  # Replace with your product category
COMPETITORS = ['Sony', 'JBL', 'Anker Soundcore']  # Replace with your competitors

# === API Keys (use Secrets in Colab) ===
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')

# GEO test query templates
QUERIES = [
    f'best {PRODUCT_CATEGORY} 2026',
    f'recommend {PRODUCT_CATEGORY} for daily use',
    f'{PRODUCT_CATEGORY} buying guide',
    f'compare {PRODUCT_CATEGORY} options under $100',
    f'top rated {PRODUCT_CATEGORY} review',
]

print(f'Brand: {BRAND_NAME}')
print(f'Category: {PRODUCT_CATEGORY}')
print(f'Competitors: {COMPETITORS}')
print(f'Test queries: {len(QUERIES)}')

## 3. ChatGPT Visibility Test

In [ ]:
from openai import OpenAI
import pandas as pd

client = OpenAI(api_key=OPENAI_API_KEY)

def test_chatgpt_visibility(query, brand, competitors):
    """Test if brand is mentioned in ChatGPT response"""
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': query}],
            max_tokens=500
        )
        answer = response.choices[0].message.content.lower()
        
        result = {
            'query': query,
            'platform': 'ChatGPT',
            'brand_mentioned': brand.lower() in answer,
            'brand_position': answer.find(brand.lower()) if brand.lower() in answer else -1,
            'competitors_mentioned': {c: c.lower() in answer for c in competitors},
            'response_preview': answer[:200]
        }
        return result
    except Exception as e:
        return {'query': query, 'platform': 'ChatGPT', 'error': str(e)}

# Run tests
results = []
for q in QUERIES:
    print(f'Testing: {q}')
    r = test_chatgpt_visibility(q, BRAND_NAME, COMPETITORS)
    results.append(r)
    mentioned = '✅' if r.get('brand_mentioned') else '❌'
    print(f'  Brand mentioned: {mentioned}')
    for comp, found in r.get('competitors_mentioned', {}).items():
        print(f'  {comp}: {"✅" if found else "❌"}')
    print()

## 4. Visibility Score

In [ ]:
# Calculate visibility scores
brand_mentions = sum(1 for r in results if r.get('brand_mentioned', False))
total_queries = len(QUERIES)
brand_score = brand_mentions / total_queries * 100

comp_scores = {}
for comp in COMPETITORS:
    mentions = sum(1 for r in results if r.get('competitors_mentioned', {}).get(comp, False))
    comp_scores[comp] = mentions / total_queries * 100

print('=== GEO Visibility Scorecard ===')
print(f'\n{BRAND_NAME}: {brand_score:.0f}% ({brand_mentions}/{total_queries} queries)')
for comp, score in sorted(comp_scores.items(), key=lambda x: -x[1]):
    print(f'{comp}: {score:.0f}%')

# Verdict
if brand_score >= 60:
    print(f'\n✅ Good AI visibility! Your brand is well-represented.')
elif brand_score >= 20:
    print(f'\n⚠️ Moderate visibility. Room for improvement.')
else:
    print(f'\n❌ Low visibility. GEO optimization needed urgently.')

## 5. Gap Analysis & Recommendations

In [ ]:
print('=== Gap Analysis ===')
print(f'\nYour brand visibility: {brand_score:.0f}%')
best_comp = max(comp_scores, key=comp_scores.get)
print(f'Best competitor ({best_comp}): {comp_scores[best_comp]:.0f}%')
gap = comp_scores[best_comp] - brand_score
print(f'Gap: {gap:.0f} percentage points')

print('\n=== Recommended Actions ===')
actions = []
if brand_score < 40:
    actions.append('P0: Add complete Product Schema (JSON-LD) to your Shopify/website')
    actions.append('P0: Create FAQ page with 10+ questions (with FAQ Schema)')
if brand_score < 60:
    actions.append('P1: Get 3+ third-party reviews/mentions on authority sites')
    actions.append('P1: Create comparison content (your brand vs competitors)')
actions.append('P2: Ensure 50+ customer reviews on Amazon/Trustpilot')
actions.append('P2: Enable Shopify UCP protocol for Agentic Commerce')

for i, a in enumerate(actions, 1):
    print(f'{i}. {a}')

# Export
report = {
    'date': datetime.now().isoformat(),
    'brand': BRAND_NAME,
    'brand_score': brand_score,
    'competitor_scores': comp_scores,
    'gap': gap,
    'actions': actions
}
with open('geo_audit_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nReport saved to geo_audit_report.json')